# Phase 3+4 integration — n=10 verification with Saighi A_k self-inhibition (Colab)

Runs `experiments/19_phase34_integrated.py` for **10 seeds** {17, 11, 23, 1, 2, 3, 5, 7, 13, 19} on CUDA with `--inhibition-gain 0.01 --n-cues 3000`.

**Why n=10 and n_cues=3000:**
- Report 032 already showed n=5 has been *sample-lucky* twice (029, 034-seed-1). The variance bar to actually close blocker #2 is n=10 with the same 10 seeds report 032 used.
- Report 033 telemetry: patterns only reach equilibrium at cue ~1200, so n_cues=1500 cuts off right after equilibrium. n_cues=3000 gives 1500 cues *of equilibrium dynamics* — more sensitive to A_k effects, more representative of architectural behavior.

**Verifies report 034's seed-1 result on a serious n.** Single-seed showed all three Phase 4 headlines improving:
- top1 +0.009 (REVERSES blocker #6' direction)
- R@10 +0.022
- cap_t05 +0.022

Telemetry matched Saighi's prediction (only 3/4097 over-used patterns accumulated inhibition).

**What we're watching:**
- 8/10+ seeds with ΔR@10 ≥ 0 vs no-Phase-4 baseline (C - A), CI disjoint from zero
- Seed 23 specifically (persistent negative outlier — most informative single test)
- Δtop1 distribution (should not regress; ideally positive across seeds)
- Δcap_t05 — the second-headline that's been blocker #3 since report 026
- A_k accumulation pattern in `death_diag` (only over-used patterns get inhibited)

Output paths use `saighi_n3k` tag so they don't collide with prior runs.

Estimated runtime: ~60-90 min on A100 (6-9 min/seed × 10 seeds, 3 conditions per seed).

In [ ]:
# 1. Clone the repo. The Saighi A_k code must be pushed to main first.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

In [ ]:
# 2. Mount Drive and copy the codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps. Colab ships torch+CUDA; we just need datasets (for wikitext).
!pip install -q datasets

In [ ]:
# 4. Sanity check: GPU present, FHRR substrate + codebook load, A_k mechanism wired.
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

import sys; sys.path.insert(0, 'src')
from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cuda')
print('codebook:', cb.shape, cb.dtype, cb.device)

# Verify A_k mechanism is present in this checkout (otherwise the run will
# behave like baseline despite --inhibition-gain).
from energy_memory.phase4.consolidation import ConsolidationConfig, ConsolidationState
cfg = ConsolidationConfig(m=4, alpha=0.25, inhibition_gain=0.01)
s = ConsolidationState(cfg, device='cuda')
s.add_pattern()
s.accumulate_inhibition(0)
assert abs(s.A[0].item() - 0.01) < 1e-6, 'A_k not wired — code is stale, abort!'
print('A_k mechanism present, accumulate_inhibition works:', s.A[0].item())

In [ ]:
# 5. Run 10-seed sweep with --inhibition-gain 0.01 --n-cues 3000.
import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]   # same 10 as report 032
log_root = Path(f'reports/phase34_{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for seed in SEEDS:
    out_dir = f'reports/phase34_{RUN_TAG}_seed{seed}'
    log_path = log_root / f'seed{seed}.log'
    print(f'\n=== seed {seed} -> {out_dir} ===  ({time.strftime("%H:%M:%S")})')
    with open(log_path, 'w') as logf:
        proc = subprocess.run(
            ['python', 'experiments/19_phase34_integrated.py',
             '--updater-kind', 'hebbian',
             '--seed', str(seed),
             '--n-cues', '3000',
             '--checkpoint-every', '500',
             '--success-threshold', '0.3',
             '--death-threshold', '0.05',
             '--death-window', '10',
             '--inhibition-gain', '0.01',
             '--inhibition-decay', '0.0',
             '--no-reencode-discovered',
             '--output-dir', out_dir],
            stdout=logf, stderr=subprocess.STDOUT,
        )
    print(f'  exit={proc.returncode}  ({time.strftime("%H:%M:%S")}, elapsed {(time.time()-t0)/60:.1f} min)')
    !tail -5 {log_path}
print(f'\nTOTAL: {(time.time()-t0)/60:.1f} min')

In [ ]:
# 6. Copy results back to Drive. Claude will pull from there for analysis.
import shutil, os
RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for seed in SEEDS:
    src = f'reports/phase34_{RUN_TAG}_seed{seed}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/phase34_{RUN_TAG}_seed{seed}', dirs_exist_ok=True)
shutil.copytree(f'reports/phase34_{RUN_TAG}_colab',
                f'{dst}/phase34_{RUN_TAG}_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst} | grep -E 'saighi_n3k'

In [ ]:
# 7. In-notebook summary table (per-seed + aggregate). Claude will re-do this with the raw JSONs but this lets you spot anomalies live.
import json
from pathlib import Path
import statistics as st

RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
rows = []
print(f'{"seed":>4} {"cond":>16} {"top1":>8} {"R@10":>8} {"cap_t05":>8} {"cands":>6} {"cons":>5} {"deaths":>6} {"A_max":>8} {"A_nz":>5}')
for seed in SEEDS:
    p = Path(f'reports/phase34_{RUN_TAG}_seed{seed}/phase34_results.json')
    if not p.exists():
        print(f'{seed:>4} MISSING')
        continue
    d = json.loads(p.read_text())
    A = B = C = None
    for cond in ('baseline_static', 'phase3_reencode', 'phase3_phase4'):
        f = d['results'][cond][-1]
        cands = f.get('candidates_total', 0)
        cons_ = f.get('consolidations', 0)
        deaths = f.get('deaths_total', 0)
        dd = f.get('death_diag', {}).get('2', {})
        A_max = dd.get('inhibition_max', 0.0)
        A_nz  = dd.get('inhibition_nonzero', 0)
        print(f'{seed:>4} {cond:>16} {f["top1"]:>8.4f} {f["topk"]:>8.4f} {f["cap_t_05"]:>8.4f} '
              f'{cands:>6} {cons_:>5} {deaths:>6} {A_max:>8.3f} {A_nz:>5}')
        if cond == 'baseline_static':  A = f
        elif cond == 'phase3_reencode': B = f
        elif cond == 'phase3_phase4':   C = f
    if A and B and C:
        rows.append({'seed': seed,
                     'dR10_CA': C['topk']-A['topk'],
                     'dR10_CB': C['topk']-B['topk'],
                     'dtop1_CA': C['top1']-A['top1'],
                     'dcap_CA': C['cap_t_05']-A['cap_t_05']})

print()
print(f'=== aggregate across {len(rows)} seeds ===')
# t-critical 95%: 9df=2.262, 8df=2.306, 4df=2.776
tcrit = {9: 2.262, 8: 2.306, 7: 2.365, 6: 2.447, 5: 2.571, 4: 2.776}
df = max(1, len(rows)-1)
tc = tcrit.get(df, 2.262)
for k in ['dR10_CA','dR10_CB','dtop1_CA','dcap_CA']:
    vals = [r[k] for r in rows]
    m = st.mean(vals); sd = st.stdev(vals) if len(vals)>1 else 0.0
    se = sd / (len(vals)**0.5) if len(vals)>1 else 0.0
    lo, hi = m - tc*se, m + tc*se
    pos = sum(1 for v in vals if v > 0)
    print(f'  {k}: mean={m:+.4f}, 95%CI [{lo:+.4f}, {hi:+.4f}], pos/total={pos}/{len(rows)}, '
          f'per-seed={[f"{v:+.4f}" for v in vals]}')